# English-Farsi Neural Machine Translation with RNN + Attention

This notebook demonstrates building a sequence-to-sequence (seq2seq) model with attention for English-Farsi translation using the OPUS-100 dataset.

## 1. Import Libraries and Setup

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import tarfile
import os
import glob
import nltk
from collections import Counter
import re
import random
import numpy as np

nltk.download('punkt', quiet=True)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

## 2. Extract Dataset and Locate Files

In [3]:
tar_path = r"D:\Database & Datasets\opus-100-corpus-en-fa-v1.0.tar.gz"
extract_to = r"D:\Database & Datasets\opus-100-en-fa"

# Extract if not already done
if not os.path.exists(extract_to) or not os.listdir(extract_to):
    print("Extracting dataset...")
    os.makedirs(extract_to, exist_ok=True)
    with tarfile.open(tar_path, "r:gz") as tar:
        tar.extractall(path=extract_to)
    print("Extraction completed.")

# Auto-discover .en and .fa files
en_files = glob.glob(os.path.join(extract_to, "**", "*.en"), recursive=True)
fa_files = glob.glob(os.path.join(extract_to, "**", "*.fa"), recursive=True)

if not en_files:
    raise FileNotFoundError("No .en file found in extracted dataset!")
if not fa_files:
    raise FileNotFoundError("No .fa file found in extracted dataset!")

EN_FILE = en_files[0]
FA_FILE = fa_files[0]

print(f"Using English file: {EN_FILE}")
print(f"Using Farsi file:   {FA_FILE}")

Using English file: D:\Database & Datasets\opus-100-en-fa\opus-100-corpus\v1.0\supervised\en-fa\opus.en-fa-dev.en
Using Farsi file:   D:\Database & Datasets\opus-100-en-fa\opus-100-corpus\v1.0\supervised\en-fa\opus.en-fa-dev.fa


## 3. Preview Data

In [4]:
print("\n--- Sample from English file ---")
with open(EN_FILE, 'r', encoding='utf-8') as f:
    for i, line in enumerate(f):
        if i >= 3: break
        print(f"EN {i+1}: {line.strip()}")

print("\n--- Sample from Farsi file ---")
with open(FA_FILE, 'r', encoding='utf-8') as f:
    for i, line in enumerate(f):
        if i >= 3: break
        print(f"FA {i+1}: {line.strip()}")


--- Sample from English file ---
EN 1: And they say: There is naught but our life of the world; we die and we live, and naught destroyeth us save time; when they have no knowledge whatsoever of (all) that; they do but guess.
EN 2: I send you out for exciting new designer drugs, and you come back with tomato sauce.
EN 3: - I don't eat meat or fish.

--- Sample from Farsi file ---
FA 1: و گفتند: جز زندگى دنيوى ما هيچ نيست. مى‌ميريم و زنده مى‌شويم و ما را جز دهر هلاك نكند. آنان را بدان دانشى نيست و جز در پندارى نيستند.
FA 2: من فرستادمتون که يه نوع مواد مخدر جديد کشف کنيد و شما با سس گوجه فرنگي برگشتين
FA 3: - ماهي و گوشت نميخورم .


## 4. Preprocessing Functions and Load Sentences

In [5]:
def preprocess_sentence(sentence, lang='en'):
    sentence = sentence.strip().lower()
    # Add space around punctuation
    sentence = re.sub(r"([?.!,¿،؛])", r" \1 ", sentence)
    sentence = re.sub(r'[" "]+', " " , sentence)
    if lang == 'en':
        sentence = re.sub(r"[^a-zA-Z?.!,¿]+", " " , sentence)
    else:  # Persian
        sentence = re.sub(r"[^\u0600-\u06FF\s?.!,،؛]+", " " , sentence)
    sentence = sentence.strip()
    return sentence

def load_sentences(en_file, fa_file, max_len=20, max_pairs=30000):
    en_sentences = []
    fa_sentences = []
    with open(en_file, 'r', encoding='utf-8') as fen, open(fa_file, 'r', encoding='utf-8') as ffa:
        for i, (en_line, fa_line) in enumerate(zip(fen, ffa)):
            if i >= max_pairs:
                break
            en_line = preprocess_sentence(en_line, 'en')
            fa_line = preprocess_sentence(fa_line, 'fa')
            if 1 <= len(en_line.split()) <= max_len and 1 <= len(fa_line.split()) <= max_len:
                en_sentences.append(en_line)
                fa_sentences.append(fa_line)
    print(f"Loaded {len(en_sentences)} valid sentence pairs.")
    return en_sentences, fa_sentences

print("\nLoading and preprocessing sentences...")
en_sents, fa_sents = load_sentences(EN_FILE, FA_FILE, max_len=15, max_pairs=30000)


Loading and preprocessing sentences...
Loaded 1576 valid sentence pairs.


## 5. Build Vocabularies

In [6]:
class Vocab:
    def __init__(self, sentences, max_vocab_size=10000):
        self.word2idx = {'<PAD>': 0, '<SOS>': 1, '<EOS>': 2, '<UNK>': 3}
        self.idx2word = {0: '<PAD>', 1: '<SOS>', 2: '<EOS>', 3: '<UNK>'}
        self.word_freq = Counter()
        
        for sent in sentences:
            for word in sent.split():
                self.word_freq[word] += 1
        
        for word, _ in self.word_freq.most_common(max_vocab_size - 4):
            idx = len(self.word2idx)
            self.word2idx[word] = idx
            self.idx2word[idx] = word

    def encode(self, sentence):
        tokens = ['<SOS>'] + sentence.split() + ['<EOS>']
        return [self.word2idx.get(word, self.word2idx['<UNK>']) for word in tokens]

    def decode(self, indices):
        words = [self.idx2word.get(idx, '<UNK>') for idx in indices]
        return ' '.join(words)

print("Building vocabularies...")
en_vocab = Vocab(en_sents, max_vocab_size=8000)
fa_vocab = Vocab(fa_sents, max_vocab_size=8000)

print(f"English vocab size: {len(en_vocab.word2idx)}")
print(f"Farsi vocab size:   {len(fa_vocab.word2idx)}")

Building vocabularies...
English vocab size: 2422
Farsi vocab size:   3526


## 6. Create Dataset and DataLoader

In [7]:
class TranslationDataset(Dataset):
    def __init__(self, en_sents, fa_sents, en_vocab, fa_vocab):
        self.en_sents = en_sents
        self.fa_sents = fa_sents
        self.en_vocab = en_vocab
        self.fa_vocab = fa_vocab

    def __len__(self):
        return len(self.en_sents)

    def __getitem__(self, idx):
        en_encoded = self.en_vocab.encode(self.en_sents[idx])
        fa_encoded = self.fa_vocab.encode(self.fa_sents[idx])
        return torch.tensor(en_encoded), torch.tensor(fa_encoded)

def collate_fn(batch):
    en_batch, fa_batch = zip(*batch)
    en_batch = nn.utils.rnn.pad_sequence(en_batch, batch_first=True, padding_value=0)
    fa_batch = nn.utils.rnn.pad_sequence(fa_batch, batch_first=True, padding_value=0)
    return en_batch, fa_batch

dataset = TranslationDataset(en_sents, fa_sents, en_vocab, fa_vocab)
dataloader = DataLoader(dataset, batch_size=64, shuffle=True, collate_fn=collate_fn)

print(f"DataLoader ready with {len(dataset)} samples.")

DataLoader ready with 1576 samples.


## 7. Define Model Architecture

In [8]:
class Encoder(nn.Module):
    def __init__(self, vocab_size, embed_size, hidden_size, num_layers=1, dropout=0.1):
        super(Encoder, self).__init__()
        self.num_layers = num_layers
        self.embedding = nn.Embedding(vocab_size, embed_size, padding_idx=0)
        self.rnn = nn.GRU(embed_size, hidden_size, num_layers, batch_first=True, dropout=dropout, bidirectional=True)
        self.fc = nn.Linear(hidden_size * 2, hidden_size)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        embedded = self.dropout(self.embedding(x))  # [B, L, E]
        outputs, hidden = self.rnn(embedded)       # outputs: [B, L, H*2], hidden: [2*num_layers, B, H]

        # Ensure hidden is always [2*num_layers, B, H]
        if hidden.dim() == 2:  # Case for batch_size=1
            hidden = hidden.unsqueeze(1)  # [2*num_layers, 1, H]

        # Extract last forward and backward hidden states
        h_forward = hidden[-2, :, :]  # [B, H]
        h_backward = hidden[-1, :, :]  # [B, H]

        # Concatenate forward and backward hidden states
        hidden_cat = torch.cat([h_forward, h_backward], dim=-1)  # [B, 2*H]

        # Apply fully connected layer
        hidden = torch.tanh(self.fc(hidden_cat))  # [B, H]

        # Reshape hidden to [1, B, H] for decoder compatibility
        hidden = hidden.unsqueeze(0)  # [1, B, H]

        # Repeat hidden state for each layer of the decoder
        hidden = hidden.repeat(self.num_layers, 1, 1)  # [num_layers, B, H]

        return outputs, hidden

class Attention(nn.Module):
    def __init__(self, hidden_size):
        super(Attention, self).__init__()
        self.Wa = nn.Linear(hidden_size * 2, hidden_size)  # encoder is bi-dir -> H*2
        self.Ua = nn.Linear(hidden_size, hidden_size)      # decoder hidden H
        self.Va = nn.Linear(hidden_size, 1)

    def forward(self, encoder_outputs, decoder_hidden):
        # encoder_outputs: [B, L, H*2]
        # decoder_hidden: [num_layers, B, H]  (we use the top layer)
        seq_len = encoder_outputs.size(1)

        # Take the last layer of decoder_hidden (shape -> [B, H])
        dec_top = decoder_hidden[-1]                 # [B, H]
        dec_top = dec_top.unsqueeze(1)               # [B, 1, H]
        dec_top = dec_top.repeat(1, seq_len, 1)      # [B, L, H]

        energy = torch.tanh(self.Wa(encoder_outputs) + self.Ua(dec_top))  # [B, L, H]
        attention = self.Va(energy).squeeze(2)                            # [B, L]
        return torch.softmax(attention, dim=1)                            # [B, L]


class Decoder(nn.Module):
    def __init__(self, vocab_size, embed_size, hidden_size, attention, num_layers=1, dropout=0.1):
        super(Decoder, self).__init__()
        self.vocab_size = vocab_size
        self.embedding = nn.Embedding(vocab_size, embed_size, padding_idx=0)
        self.attention = attention
        self.rnn = nn.GRU(embed_size + hidden_size * 2, hidden_size, num_layers, batch_first=True, dropout=dropout)
        self.fc_out = nn.Linear(hidden_size + hidden_size * 2, vocab_size)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, hidden, encoder_outputs):
        x = x.unsqueeze(1)  # [B, 1]
        embedded = self.dropout(self.embedding(x))  # [B, 1, E]
        a = self.attention(encoder_outputs, hidden)  # [B, L]
        a = a.unsqueeze(1)  # [B, 1, L]
        context = torch.bmm(a, encoder_outputs)  # [B, 1, H*2]
        rnn_input = torch.cat((embedded, context), dim=2)  # [B, 1, E + H*2]
        output, hidden = self.rnn(rnn_input, hidden)       # output: [B, 1, H], hidden: [1, B, H]
        output = torch.cat((output.squeeze(1), context.squeeze(1)), dim=1)  # [B, H + H*2]
        output = self.fc_out(output)  # [B, vocab_size]
        return output, hidden

class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super(Seq2Seq, self).__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device

    def forward(self, src, trg, teacher_forcing_ratio=0.5):
        batch_size = src.shape[0]
        trg_len = trg.shape[1]
        trg_vocab_size = self.decoder.vocab_size

        outputs = torch.zeros(batch_size, trg_len, trg_vocab_size).to(self.device)
        encoder_outputs, hidden = self.encoder(src)

        x = trg[:, 0]  # <SOS> token
        for t in range(1, trg_len):
            output, hidden = self.decoder(x, hidden, encoder_outputs)
            outputs[:, t, :] = output
            teacher_force = random.random() < teacher_forcing_ratio
            top1 = output.argmax(1)
            x = trg[:, t] if teacher_force else top1

        return outputs

## 8. Training Setup

In [9]:
INPUT_DIM = len(en_vocab.word2idx)
OUTPUT_DIM = len(fa_vocab.word2idx)
EMBED_SIZE = 256
HIDDEN_SIZE = 512
NUM_LAYERS = 2
DROPOUT = 0.3
LR = 0.001
EPOCHS = 10

attention = Attention(HIDDEN_SIZE)
encoder = Encoder(INPUT_DIM, EMBED_SIZE, HIDDEN_SIZE, NUM_LAYERS, DROPOUT)
decoder = Decoder(OUTPUT_DIM, EMBED_SIZE, HIDDEN_SIZE, attention, NUM_LAYERS, DROPOUT)
model = Seq2Seq(encoder, decoder, device).to(device)

optimizer = optim.Adam(model.parameters(), lr=LR)
criterion = nn.CrossEntropyLoss(ignore_index=0)  # ignore <PAD>

## 9. Training Loop

In [10]:
def train(model, dataloader, optimizer, criterion, clip=1.0):
    model.train()
    epoch_loss = 0
    for i, (src, trg) in enumerate(dataloader):
        src, trg = src.to(device), trg.to(device)
        optimizer.zero_grad()
        output = model(src, trg)  # [B, T, V]
        output_dim = output.shape[-1]
        output = output[:, 1:].reshape(-1, output_dim)  # skip <SOS>
        trg = trg[:, 1:].reshape(-1)  # skip <SOS>
        loss = criterion(output, trg)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
        optimizer.step()
        epoch_loss += loss.item()
        if i % 100 == 0:
            print(f"  Batch {i:4d} | Loss: {loss.item():.4f}")
    return epoch_loss / len(dataloader)

print(f"Starting training on {device}...")
for epoch in range(EPOCHS):
    print(f" Epoch {epoch+1}/{EPOCHS} ".center(50, "="))
    loss = train(model, dataloader, optimizer, criterion)
print(f"Epoch {epoch+1} | Average Loss: {loss:.4f}")
print("Training completed!")
print("Training completed!")

Starting training on cpu...
=================== Epoch 1/10 ===================
  Batch    0 | Loss: 8.1700
  Batch    0 | Loss: 8.1700
=================== Epoch 2/10 ===================
=================== Epoch 2/10 ===================
  Batch    0 | Loss: 6.0151
  Batch    0 | Loss: 6.0151
=================== Epoch 3/10 ===================
=================== Epoch 3/10 ===================
  Batch    0 | Loss: 5.5535
  Batch    0 | Loss: 5.5535
=================== Epoch 4/10 ===================
=================== Epoch 4/10 ===================
  Batch    0 | Loss: 4.8367
  Batch    0 | Loss: 4.8367
=================== Epoch 5/10 ===================
=================== Epoch 5/10 ===================
  Batch    0 | Loss: 3.6269
  Batch    0 | Loss: 3.6269
=================== Epoch 6/10 ===================
=================== Epoch 6/10 ===================
  Batch    0 | Loss: 2.9505
  Batch    0 | Loss: 2.9505
=================== Epoch 7/10 ===================
=================== Epoc

## 10. Translation Function

In [11]:
def translate_sentence(sentence, model, en_vocab, fa_vocab, device, max_len=50):
    model.eval()
    with torch.no_grad():
        tokens = en_vocab.encode(sentence)
        src_tensor = torch.LongTensor(tokens).unsqueeze(0).to(device)
        encoder_outputs, hidden = model.encoder(src_tensor)

        trg_indexes = [fa_vocab.word2idx['<SOS>']]
        for i in range(max_len):
            trg_tensor = torch.LongTensor([trg_indexes[-1]]).to(device)
            output, hidden = model.decoder(trg_tensor, hidden, encoder_outputs)
            pred_token = output.argmax(1).item()
            trg_indexes.append(pred_token)
            if pred_token == fa_vocab.word2idx['<EOS>']:
                break

        trg_tokens = [fa_vocab.idx2word.get(i, '<UNK>') for i in trg_indexes]
        # Remove <SOS> and <EOS>
        translation = ' '.join(trg_tokens[1:-1])
        return translation

## 11. Test Translation

In [12]:
test_sentences = [
    "Hello, how are you?",
    "I love machine learning.",
    "What is your name?",
    "Thank you very much."
]

print("TRANSLATION TEST".center(50, "="))
for sent in test_sentences:
    translated = translate_sentence(sent, model, en_vocab, fa_vocab, device)
    print(f"EN: {sent}")
print(f"FA: {translated}\n")

print("Done!")

=================TRANSLATION TEST=================
EN: Hello, how are you?
EN: I love machine learning.
EN: What is your name?
EN: Thank you very much.
FA: مي تو رو رو رو رو رو کني

Done!
